In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.interpolate import splprep, splev

try:
    from skimage.exposure import rescale_intensity
    from skimage.filters import gaussian
    from skimage.segmentation import active_contour
except ImportError as exc:
    raise ImportError(
        "This notebook requires scikit-image in the current kernel. Install it with `pip install scikit-image`."
    ) from exc


data_dir = Path.cwd()
SHOW_PLOTS = False
TESTING_MODE = False
DYNAMIC_CONTOUR_CACHE_DIR = data_dir / "dynamic_contour_cache"


def make_path(*parts):
    return data_dir.joinpath(*parts)


def make_dataset_config(image_path, polyline_path, spots_path, contour_preset, dynamic_anchor_frame_interval=1):
    return {
        "image_path": Path(image_path),
        "polyline_path": Path(polyline_path),
        "spots_path": Path(spots_path),
        "contour_preset": contour_preset,
        "dynamic_anchor_frame_interval": dynamic_anchor_frame_interval,
        "use_dynamic_contour_cache": True,
        "dynamic_contour_cache_dir": DYNAMIC_CONTOUR_CACHE_DIR,
    }


DATA_INPUT_CONFIGS = {
    "PBS": make_dataset_config(
        make_path("PBS 3-MaxIP_with brigthfield.tif"),
        make_path("PBS3_Brightfield_Frame1_Boundary_XY_Coordinates.csv"),
        make_path("PBS 3-MaxIP_segmented_spots.csv"),
        contour_preset="pbs",
        dynamic_anchor_frame_interval=50,
    ),
    "FN": make_dataset_config(
        make_path("FN 3-MaxIP.tif"),
        make_path("FN3_OriginFrame_XY_Coordinates.csv"),
        make_path("FN 3-MaxIP_spots.csv"),
        contour_preset="fn",
    ),
    "FN TOP": make_dataset_config(
        make_path("FN 3-MaxIP.tif"),
        make_path("XY_Coordinates_top.csv"),
        make_path("FN 3-MaxIP_spots.csv"),
        contour_preset="fn",
    ),
}


def add_timelapse_dataset_configs(batch_id, timelapse_dir, treatment, sample_numbers):
    contour_preset = "fn" if treatment == "FN" else "pbs"

    for sample_number in sample_numbers:
        stem = f"{batch_id}_{treatment} {sample_number}"
        image_path = make_path(timelapse_dir, f"{stem}_brightfield.tif")
        spots_path = make_path(timelapse_dir, f"{stem}_spots.csv")

        for side in ("TOP", "BOTTOM"):
            data_input = f"{stem} {side}"
            side_title = side.title()
            specific_contour_preset = "fn2" if data_input == "18951_FN 2 TOP" else contour_preset
            DATA_INPUT_CONFIGS[data_input] = make_dataset_config(
                image_path,
                make_path(timelapse_dir, f"{stem}_Coordinates_{side_title}.csv"),
                spots_path,
                contour_preset=specific_contour_preset,
            )


add_timelapse_dataset_configs("18951", "Timelapse 12", "FN", range(1, 5))
add_timelapse_dataset_configs("18951", "Timelapse 12", "PBS", range(1, 5))
add_timelapse_dataset_configs("18865", "Timelapse 11", "FN", range(5, 9))
add_timelapse_dataset_configs("18865", "Timelapse 11", "PBS", range(5, 8))

# Runs every configured DATA_INPUT by default. Edit this list to run a subset.
DATA_INPUTS = list(DATA_INPUT_CONFIGS)
REQUIRED_PATH_KEYS = ("image_path", "polyline_path", "spots_path")


In [ ]:
def build_snake_active_contour_kwargs(contour_preset, boundary_condition):
    contour_presets = {
        "pbs": dict(
            beta=0.01,
            w_line=0.0,
            w_edge=2.0,
            max_num_iter=2500,
            convergence=0.01,
        ),
        "fn": dict(
            beta=0.01,
            w_line=0.0,
            w_edge=2.0,
            max_num_iter=2500,
            convergence=0.01,
        ),
        "fn2": dict(
            beta=0.01,
            w_line=3.0,
            w_edge=2.0,
            max_num_iter=2500,
            convergence=0.01,
        ),
    }

    try:
        kwargs = contour_presets[contour_preset].copy()
    except KeyError as exc:
        raise ValueError(f"Unknown contour preset: {contour_preset}") from exc

    kwargs["boundary_condition"] = boundary_condition
    return kwargs


def missing_required_paths(config, required_path_keys=REQUIRED_PATH_KEYS):
    missing_paths = []
    for key in required_path_keys:
        path = Path(config[key])
        if not path.exists():
            missing_paths.append((key, path))
    return missing_paths


def print_missing_file_warning(data_input, missing_paths):
    missing_text = "; ".join(f"{key}: {path}" for key, path in missing_paths)
    print(f"Warning: skipping {data_input}; missing required file(s): {missing_text}")


def parse_imagej_description(text):
    metadata = {}
    for line in text.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            metadata[key.strip()] = value.strip()
    return metadata


def page_index(channel_idx, time_idx, n_channels):
    return time_idx * n_channels + channel_idx


def load_channel_frame(image_path, channel_idx, time_idx, n_channels):
    with Image.open(image_path) as tif:
        tif.seek(page_index(channel_idx, time_idx, n_channels))
        return np.asarray(tif, dtype=np.float32)


def normalize_for_display(image, vmin=None, vmax=None, lower_percentile=1, upper_percentile=99.5):
    source = image[image > 0]
    if source.size == 0:
        source = image.ravel()

    if vmin is None:
        vmin = float(np.percentile(source, lower_percentile))
    if vmax is None:
        vmax = float(np.percentile(source, upper_percentile))
    if vmax <= vmin:
        vmax = vmin + 1.0

    scaled = np.clip((image - vmin) / (vmax - vmin), 0, 1)
    return scaled, float(vmin), float(vmax)


def preprocess_snake_source(source_image):
    source_nonzero = source_image[source_image > 0]
    source_intensity_range = (
        float(np.percentile(source_nonzero, 1)) if source_nonzero.size else float(source_image.min()),
        float(np.percentile(source_nonzero, 99.5)) if source_nonzero.size else float(source_image.max()),
    )
    snake_ready = rescale_intensity(source_image, in_range=source_intensity_range, out_range=(0.0, 1.0))
    snake_ready = gaussian(snake_ready, sigma=2.0, preserve_range=True)
    return snake_ready


def resample_open_polyline(points_xy, n_points=300, smoothing=0.0):
    tck, _ = splprep([points_xy[:, 0], points_xy[:, 1]], s=smoothing, per=False)
    u = np.linspace(0, 1, n_points, endpoint=True)
    x_new, y_new = splev(u, tck)
    return np.column_stack([x_new, y_new])


def build_dynamic_anchor_frames(n_timepoints, frame_interval):
    if frame_interval <= 0:
        raise ValueError("`frame_interval` must be a positive integer.")
    if n_timepoints <= 0:
        raise ValueError("`n_timepoints` must be positive.")

    # Match the previous convention: interval 50 means frames 0, 49, 99, ... in 0-based CSV coordinates.
    interval_anchor_frames = np.arange(frame_interval - 1, n_timepoints, frame_interval, dtype=int)
    anchor_frames = np.unique(np.r_[0, interval_anchor_frames, n_timepoints - 1]).astype(int)
    return anchor_frames


def file_fingerprint(path):
    path = Path(path)
    stat = path.stat()
    return {
        "name": path.name,
        "path": str(path.resolve()),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def build_dynamic_contour_cache_metadata(
    data_input,
    image_path,
    polyline_path,
    snake_channel_index,
    n_channels,
    n_timepoints,
    dynamic_anchor_frame_interval,
    dynamic_anchor_frames,
    snake_active_contour_kwargs,
    initial_snake_xy,
):
    return json_ready(
        {
            "data_input": data_input,
            "image": file_fingerprint(image_path),
            "polyline": file_fingerprint(polyline_path),
            "snake_channel_index": snake_channel_index,
            "n_channels": n_channels,
            "n_timepoints": n_timepoints,
            "dynamic_anchor_frame_interval": dynamic_anchor_frame_interval,
            "dynamic_anchor_frames": dynamic_anchor_frames,
            "active_contour_kwargs": snake_active_contour_kwargs,
            "initial_snake_point_count": len(initial_snake_xy),
        }
    )


def dynamic_contour_cache_path(cache_dir, metadata):
    metadata_json = json.dumps(metadata, sort_keys=True)
    metadata_hash = hashlib.sha256(metadata_json.encode("utf-8")).hexdigest()[:16]
    image_stem = Path(metadata["image"]["name"]).stem
    safe_stem = "".join(char if char.isalnum() else "_" for char in image_stem).strip("_")
    return Path(cache_dir) / f"{safe_stem}_dynamic_contours_{metadata_hash}.npz"


def prepare_curve(curve_xy):
    curve = np.asarray(curve_xy, dtype=float).copy()

    if len(curve) < 2:
        raise ValueError("The boundary polyline needs at least two points.")

    segment_steps = np.linalg.norm(np.diff(curve, axis=0), axis=1)
    curve = curve[np.r_[True, segment_steps > 0]]

    if len(curve) < 2:
        raise ValueError("The boundary polyline collapses to fewer than two unique points.")

    segment_vectors = np.diff(curve, axis=0)
    segment_lengths = np.linalg.norm(segment_vectors, axis=1)
    arc_lengths = np.concatenate([[0.0], np.cumsum(segment_lengths)])
    return curve, arc_lengths


def build_curve_cache(curve_xy):
    curve, arc_lengths = prepare_curve(curve_xy)
    return {"curve": curve, "arc_lengths": arc_lengths}


def point_to_segment(p, a, b):
    ab = b - a
    ab2 = np.dot(ab, ab)
    if ab2 == 0:
        return a.copy(), 0.0

    t = np.dot(p - a, ab) / ab2
    t = np.clip(t, 0.0, 1.0)
    closest = a + t * ab
    return closest, t


def stable_segment_tangent(curve, segment_index, half_window=8, min_span=5.0):
    left = max(0, segment_index - half_window)
    right = min(len(curve) - 1, segment_index + 1 + half_window)

    while True:
        tangent = curve[right] - curve[left]
        tangent_length = np.linalg.norm(tangent)
        if tangent_length >= min_span:
            return tangent / tangent_length

        can_expand_left = left > 0
        can_expand_right = right < len(curve) - 1
        if not can_expand_left and not can_expand_right:
            break
        if can_expand_left:
            left -= 1
        if can_expand_right:
            right += 1

    # Fallback for very short curves: use the nearest non-zero local segment.
    for offset in range(len(curve) - 1):
        for candidate_index in (segment_index - offset, segment_index + offset):
            if 0 <= candidate_index < len(curve) - 1:
                tangent = curve[candidate_index + 1] - curve[candidate_index]
                tangent_length = np.linalg.norm(tangent)
                if tangent_length > 0:
                    return tangent / tangent_length

    raise ValueError("The boundary polyline has no non-zero tangent.")


def signed_distance_to_polyline(p, curve, arc_lengths):
    best_dist2 = np.inf
    best_closest = None
    best_segment_index = None
    best_t = None

    for i in range(len(curve) - 1):
        a, b = curve[i], curve[i + 1]
        closest, t = point_to_segment(p, a, b)
        diff = p - closest
        dist2 = np.dot(diff, diff)

        if dist2 < best_dist2:
            best_dist2 = dist2
            best_closest = closest
            best_segment_index = i
            best_t = t

    a = curve[best_segment_index]
    b = curve[best_segment_index + 1]
    tangent = stable_segment_tangent(curve, best_segment_index)
    normal = np.array([-tangent[1], tangent[0]])

    offset = p - best_closest
    dot = np.dot(offset, normal)
    sign = -1.0 if dot > 0 else 1.0 if dot < 0 else 0.0

    signed_distance = sign * np.sqrt(best_dist2)
    s = arc_lengths[best_segment_index] + best_t * np.linalg.norm(b - a)
    return signed_distance, best_closest, best_segment_index, s


def measure_points_against_curve(points_xy, curve_cache):
    if len(points_xy) == 0:
        return np.empty((0, 4), dtype=object)

    return np.array(
        [signed_distance_to_polyline(p, curve_cache["curve"], curve_cache["arc_lengths"]) for p in points_xy],
        dtype=object,
    )


def assign_dynamic_anchor_frame(frame_value, anchor_frames):
    diffs = np.abs(anchor_frames - frame_value)
    min_diff = diffs.min()
    # Later-anchor tie breaking: if two anchors are equally close, choose the later one.
    return int(anchor_frames[diffs == min_diff][-1])


def validate_dynamic_boundary_assignments(
    df_spots_dynamic,
    dynamic_anchor_frames,
    dynamic_anchor_curve_caches,
    distance_atol=1e-6,
    arc_length_atol=1e-6,
):
    expected_anchor_frames = np.array(
        [
            assign_dynamic_anchor_frame(frame_value, dynamic_anchor_frames)
            for frame_value in df_spots_dynamic["Frame"].to_numpy(dtype=int)
        ],
        dtype=int,
    )
    assigned_anchor_frames = df_spots_dynamic["dynamic_anchor_frame"].to_numpy(dtype=int)

    if not np.array_equal(assigned_anchor_frames, expected_anchor_frames):
        mismatch_count = int(np.count_nonzero(assigned_anchor_frames != expected_anchor_frames))
        raise ValueError(
            "Dynamic boundary assignment mismatch: "
            f"{mismatch_count} spot(s) were not assigned to the most temporally proximal boundary."
        )

    missing_curve_frames = sorted(set(assigned_anchor_frames) - set(dynamic_anchor_curve_caches))
    if missing_curve_frames:
        raise ValueError(f"Missing dynamic boundary curve cache(s) for anchor frame(s): {missing_curve_frames}")

    for anchor_frame in dynamic_anchor_frames:
        mask = assigned_anchor_frames == anchor_frame
        if not np.any(mask):
            continue

        anchor_points = df_spots_dynamic.loc[mask, ["X", "Y"]].to_numpy(dtype=float)
        anchor_results = measure_points_against_curve(anchor_points, dynamic_anchor_curve_caches[anchor_frame])
        recalculated_signed_distance = anchor_results[:, 0].astype(float)
        recalculated_arc_length = anchor_results[:, 3].astype(float)

        stored_signed_distance = df_spots_dynamic.loc[mask, "dynamic_signed_distance_to_boundary"].to_numpy(dtype=float)
        stored_arc_length = df_spots_dynamic.loc[mask, "dynamic_boundary_arc_length"].to_numpy(dtype=float)

        if not np.allclose(stored_signed_distance, recalculated_signed_distance, atol=distance_atol, rtol=0):
            raise ValueError(
                "Dynamic signed distance validation failed for "
                f"anchor frame {anchor_frame}; distances were not calculated from the assigned boundary."
            )
        if not np.allclose(stored_arc_length, recalculated_arc_length, atol=arc_length_atol, rtol=0):
            raise ValueError(
                "Dynamic boundary arc-length validation failed for "
                f"anchor frame {anchor_frame}; arc lengths were not calculated from the assigned boundary."
            )


def load_dynamic_anchor_curves_from_cache(cache_path, expected_anchor_frames):
    if not Path(cache_path).exists():
        return None

    with np.load(cache_path, allow_pickle=False) as cached:
        cached_anchor_frames = cached["anchor_frames"].astype(int)
        if not np.array_equal(cached_anchor_frames, expected_anchor_frames):
            return None
        cached_curves = cached["curves"].astype(float)

    return {int(anchor_frame): cached_curves[index].copy() for index, anchor_frame in enumerate(cached_anchor_frames)}


def save_dynamic_anchor_curves_to_cache(cache_path, metadata, anchor_frames, anchor_curves):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    curves = np.stack([anchor_curves[int(anchor_frame)] for anchor_frame in anchor_frames], axis=0)
    np.savez_compressed(
        cache_path,
        anchor_frames=np.asarray(anchor_frames, dtype=int),
        curves=curves,
        metadata_json=np.array(json.dumps(metadata, sort_keys=True)),
    )


def build_dynamic_anchor_assignment_check_frames(anchor_frames, n_timepoints):
    check_frames = [0, n_timepoints - 1]
    for left_anchor, right_anchor in zip(anchor_frames[:-1], anchor_frames[1:]):
        midpoint_floor = (left_anchor + right_anchor) // 2
        check_frames.extend([left_anchor, midpoint_floor, min(midpoint_floor + 1, right_anchor), right_anchor])
    return np.array(sorted(set(np.clip(check_frames, 0, n_timepoints - 1))), dtype=int)


def summarize_track_link_metric(track_links, metric_column, output_prefix):
    summary_columns = [
        "DATA_INPUT",
        "Track ID",
        f"mean_{output_prefix}_um_per_s",
        f"max_{output_prefix}_um_per_s",
        f"median_{output_prefix}_um_per_s",
        f"std_{output_prefix}_um_per_s",
    ]
    if track_links.empty:
        return pd.DataFrame(columns=summary_columns)

    return (
        track_links.groupby(["DATA_INPUT", "Track ID"])[metric_column]
        .agg(["mean", "max", "median", "std"])
        .rename(
            columns={
                "mean": f"mean_{output_prefix}_um_per_s",
                "max": f"max_{output_prefix}_um_per_s",
                "median": f"median_{output_prefix}_um_per_s",
                "std": f"std_{output_prefix}_um_per_s",
            }
        )
        .reset_index()
    )


def compute_dynamic_track_motion(df_spots_dynamic):
    required_columns = [
        "DATA_INPUT",
        "Track ID",
        "Spot ID",
        "T",
        "Frame",
        "X",
        "Y",
        "dynamic_anchor_frame",
        "dynamic_signed_distance_to_boundary",
        "dynamic_boundary_arc_length",
    ]
    missing_columns = [column for column in required_columns if column not in df_spots_dynamic.columns]
    if missing_columns:
        raise ValueError(f"Cannot compute dynamic track motion; missing columns: {missing_columns}")

    track_links = df_spots_dynamic[required_columns].copy()
    track_links = track_links.sort_values(["DATA_INPUT", "Track ID", "T", "Spot ID"]).reset_index(drop=True)

    same_track_as_next = (
        (track_links["DATA_INPUT"] == track_links["DATA_INPUT"].shift(-1))
        & (track_links["Track ID"] == track_links["Track ID"].shift(-1))
    )

    next_column_map = {
        "Spot ID": "next_spot_id",
        "T": "next_t",
        "Frame": "next_frame",
        "X": "next_x",
        "Y": "next_y",
        "dynamic_anchor_frame": "next_dynamic_anchor_frame",
        "dynamic_signed_distance_to_boundary": "next_dynamic_signed_distance",
        "dynamic_boundary_arc_length": "next_dynamic_boundary_arc_length",
    }
    for source_column, next_column in next_column_map.items():
        track_links[next_column] = track_links[source_column].shift(-1)

    track_links["delta_t_s"] = track_links["next_t"] - track_links["T"]
    track_links["delta_x_um"] = track_links["next_x"] - track_links["X"]
    track_links["delta_y_um"] = track_links["next_y"] - track_links["Y"]
    track_links["delta_dynamic_signed_distance_um"] = (
        track_links["next_dynamic_signed_distance"] - track_links["dynamic_signed_distance_to_boundary"]
    )

    track_links = track_links[same_track_as_next].copy()
    track_links = track_links[track_links["delta_t_s"] > 0].copy()

    track_links["dynamic_radial_speed_um_per_s"] = (
        track_links["delta_dynamic_signed_distance_um"] / track_links["delta_t_s"]
    )
    track_links["link_speed_um_per_s"] = (
        np.sqrt(track_links["delta_x_um"] ** 2 + track_links["delta_y_um"] ** 2) / track_links["delta_t_s"]
    )
    track_links["dynamic_tangential_speed_um_per_s"] = np.sqrt(
        np.clip(
            track_links["link_speed_um_per_s"] ** 2 - track_links["dynamic_radial_speed_um_per_s"] ** 2,
            a_min=0,
            a_max=None,
        )
    )

    dynamic_radial_speed_summary = summarize_track_link_metric(
        track_links,
        "dynamic_radial_speed_um_per_s",
        "dynamic_radial_speed",
    )
    dynamic_tangential_speed_summary = summarize_track_link_metric(
        track_links,
        "dynamic_tangential_speed_um_per_s",
        "dynamic_tangential_speed",
    )
    link_speed_summary = summarize_track_link_metric(track_links, "link_speed_um_per_s", "link_speed")

    if track_links.empty:
        dynamic_track_motion_summary = pd.DataFrame(
            columns=[
                "DATA_INPUT",
                "Track ID",
                *link_speed_summary.columns[2:],
                *dynamic_radial_speed_summary.columns[2:],
                *dynamic_tangential_speed_summary.columns[2:],
            ]
        )
    else:
        dynamic_track_motion_summary = (
            link_speed_summary.merge(dynamic_radial_speed_summary, on=["DATA_INPUT", "Track ID"], how="inner")
            .merge(dynamic_tangential_speed_summary, on=["DATA_INPUT", "Track ID"], how="inner")
        )

    return {
        "dynamic_track_links": track_links,
        "dynamic_radial_speed_summary": dynamic_radial_speed_summary,
        "dynamic_tangential_speed_summary": dynamic_tangential_speed_summary,
        "dynamic_track_motion_summary": dynamic_track_motion_summary,
    }


def plot_dynamic_distance_result(result):
    df_spots = result["df_spots"]
    df_spots_dynamic = result["df_spots_dynamic"]
    static_curve = result["static_curve"]
    dynamic_anchor_frames = result["dynamic_anchor_frames"]
    dynamic_anchor_curve_caches = result["dynamic_anchor_curve_caches"]

    combined_distances = np.concatenate(
        [
            df_spots["signed_distance_to_boundary"].to_numpy(dtype=float),
            df_spots_dynamic["dynamic_signed_distance_to_boundary"].to_numpy(dtype=float),
        ]
    )
    finite_distances = combined_distances[np.isfinite(combined_distances)]
    color_limit = float(np.nanmax(np.abs(finite_distances))) if finite_distances.size else 1.0

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

    static_scatter = axes[0].scatter(
        df_spots["X"],
        df_spots["Y"],
        c=df_spots["signed_distance_to_boundary"],
        s=10,
        cmap="coolwarm",
        alpha=0.7,
        vmin=-color_limit,
        vmax=color_limit,
    )
    axes[0].plot(static_curve[:, 0], static_curve[:, 1], color="black", linewidth=2.0, label="Static refined boundary")
    axes[0].set_aspect("equal")
    axes[0].invert_yaxis()
    axes[0].set_title(f"{result['data_input']}: static signed distance")
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Y")
    axes[0].legend(loc="best")

    dynamic_scatter = axes[1].scatter(
        df_spots_dynamic["X"],
        df_spots_dynamic["Y"],
        c=df_spots_dynamic["dynamic_signed_distance_to_boundary"],
        s=10,
        cmap="coolwarm",
        alpha=0.7,
        vmin=-color_limit,
        vmax=color_limit,
    )
    dynamic_line_colors = plt.cm.viridis(np.linspace(0, 1, len(dynamic_anchor_frames)))
    for line_color, anchor_frame in zip(dynamic_line_colors, dynamic_anchor_frames):
        dynamic_curve = dynamic_anchor_curve_caches[anchor_frame]["curve"]
        axes[1].plot(
            dynamic_curve[:, 0],
            dynamic_curve[:, 1],
            color=line_color,
            linewidth=1.5,
            label=f"Anchor frame {anchor_frame}",
        )
    axes[1].set_aspect("equal")
    axes[1].invert_yaxis()
    axes[1].set_title(f"{result['data_input']}: dynamic signed distance")
    axes[1].set_xlabel("X")
    axes[1].set_ylabel("Y")
    axes[1].legend(loc="best", fontsize=8)

    fig.colorbar(dynamic_scatter, ax=axes, label="Signed distance to boundary")
    plt.show()


TREATMENT_COLORS = {"PBS": "#008C8C", "FN": "#F28E2B", "Other": "#777777"}


def treatment_from_data_input(data_input):
    data_input_text = str(data_input).upper()
    if "PBS" in data_input_text:
        return "PBS"
    if "FN" in data_input_text:
        return "FN"
    return "Other"


def color_for_data_input(data_input):
    return TREATMENT_COLORS[treatment_from_data_input(data_input)]


def add_treatment_plot_columns(df):
    df = df.copy()
    if df.empty or "DATA_INPUT" not in df.columns:
        df["treatment"] = pd.Series(dtype=object)
        df["plot_color"] = pd.Series(dtype=object)
        return df

    df["treatment"] = df["DATA_INPUT"].apply(treatment_from_data_input)
    df["plot_color"] = df["DATA_INPUT"].apply(color_for_data_input)
    return df


def warn_if_empty_for_plot(df, plot_name):
    if df.empty:
        print(f"Skipping {plot_name}; no dynamic speed rows are available.")
        return True
    return False


def plot_cell_median_dynamic_speeds(cell_median_dynamic_speed_plot_data):
    plot_data = cell_median_dynamic_speed_plot_data.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["median_dynamic_radial_speed_um_per_s", "median_dynamic_tangential_speed_um_per_s"]
    )
    if warn_if_empty_for_plot(plot_data, "cell median dynamic speed plot"):
        return

    fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
    for treatment, treatment_data in plot_data.groupby("treatment"):
        ax.scatter(
            treatment_data["median_dynamic_radial_speed_um_per_s"],
            treatment_data["median_dynamic_tangential_speed_um_per_s"],
            s=28,
            alpha=0.75,
            color=TREATMENT_COLORS.get(treatment, TREATMENT_COLORS["Other"]),
            label=treatment,
        )

    ax.axvline(0, color="black", linewidth=0.8, alpha=0.35)
    ax.set_xlabel("Median dynamic radial speed (um/s)")
    ax.set_ylabel("Median dynamic tangential speed (um/s)")
    ax.set_title("Cell Median Dynamic Radial vs Tangential Speed")
    ax.legend(title="Input type", loc="best")
    plt.show()


def plot_data_input_timepoint_speed(timepoint_speed_plot_data, value_column, ylabel, title, plot_name):
    plot_data = timepoint_speed_plot_data.replace([np.inf, -np.inf], np.nan).dropna(subset=["T", value_column])
    if warn_if_empty_for_plot(plot_data, plot_name):
        return

    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    for data_input, data_input_rows in plot_data.groupby("DATA_INPUT"):
        data_input_rows = data_input_rows.sort_values("T")
        ax.plot(
            data_input_rows["T"],
            data_input_rows[value_column],
            marker="o",
            markersize=3,
            linewidth=1.2,
            alpha=0.85,
            color=color_for_data_input(data_input),
            label=data_input,
        )

    ax.axhline(0, color="black", linewidth=0.8, alpha=0.35)
    ax.set_xlabel("Time T (s)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title="DATA_INPUT", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=8)
    plt.show()


In [ ]:
def process_data_input(data_input, config, show_plots=False, testing_mode=False):
    image_path = Path(config["image_path"])
    polyline_path = Path(config["polyline_path"])
    spots_path = Path(config["spots_path"])
    open_snake_boundary_condition = config.get("open_snake_boundary_condition", "free")
    snake_active_contour_kwargs = build_snake_active_contour_kwargs(
        config["contour_preset"],
        open_snake_boundary_condition,
    )

    print(f"\nProcessing {data_input}")

    with Image.open(image_path) as tif:
        imagej_metadata = parse_imagej_description(str(tif.tag_v2.get(270, "")))
        n_pages = getattr(tif, "n_frames", 1)
        n_channels = int(imagej_metadata.get("channels", 1))
        n_timepoints = int(imagej_metadata.get("frames", max(1, n_pages // max(n_channels, 1))))

        channel_images_t0 = []
        for channel_idx in range(n_channels):
            tif.seek(page_index(channel_idx, 0, n_channels))
            channel_images_t0.append(np.asarray(tif, dtype=np.float32))

    channel_images_t0 = np.stack(channel_images_t0, axis=0)
    handdrawn_polyline = pd.read_csv(polyline_path)
    polyline_xy = handdrawn_polyline[["X", "Y"]].to_numpy(dtype=float)
    initial_snake_xy = resample_open_polyline(
        polyline_xy,
        n_points=config.get("initial_snake_point_count", 400),
        smoothing=config.get("initial_snake_smoothing", 0.0),
    )
    initial_snake_rc = np.column_stack([initial_snake_xy[:, 1], initial_snake_xy[:, 0]])

    manual_display_windows = config.get("manual_display_windows", {})
    channel_display_images = []
    channel_display_ranges = []
    channel_stats_rows = []

    for channel_idx, image in enumerate(channel_images_t0):
        override = manual_display_windows.get(channel_idx)
        if override is None:
            display_image, display_min, display_max = normalize_for_display(image)
        else:
            display_image, display_min, display_max = normalize_for_display(image, *override)

        nonzero = image[image > 0]
        source = nonzero if nonzero.size else image.ravel()
        channel_display_images.append(display_image)
        channel_display_ranges.append((display_min, display_max))
        channel_stats_rows.append(
            {
                "channel": channel_idx,
                "mean_intensity": float(image.mean()),
                "median_intensity": float(np.median(source)),
                "nonzero_fraction": float(np.count_nonzero(image) / image.size),
                "display_min": display_min,
                "display_max": display_max,
            }
        )

    channel_display_images = np.stack(channel_display_images, axis=0)
    channel_stats = pd.DataFrame(channel_stats_rows).set_index("channel")

    snake_channel_index = config.get("snake_channel_index")
    if snake_channel_index is None:
        snake_channel_index = int(channel_stats["median_intensity"].idxmax())

    if n_channels == 2:
        fluorescence_channel_index = 1 - snake_channel_index
    else:
        fluorescence_channel_index = 0

    brightfield_display = channel_display_images[snake_channel_index]
    fluorescence_display = channel_display_images[fluorescence_channel_index]
    composite_rgb = np.dstack(
        [
            np.clip(0.9 * brightfield_display, 0, 1),
            np.clip(0.9 * brightfield_display + fluorescence_display, 0, 1),
            np.clip(0.9 * brightfield_display, 0, 1),
        ]
    )

    snake_source = channel_images_t0[snake_channel_index]
    snake_image = preprocess_snake_source(snake_source)

    dynamic_anchor_frame_interval = int(config.get("dynamic_anchor_frame_interval", 1))
    n_timepoints_for_processing = n_timepoints
    if testing_mode:
        n_timepoints_for_processing = min(n_timepoints, 5)
        dynamic_anchor_frame_interval = 1
    dynamic_anchor_frames = build_dynamic_anchor_frames(n_timepoints_for_processing, dynamic_anchor_frame_interval)

    print(f"ImageJ reports {n_channels} channel(s), {n_timepoints} timepoint(s), and {n_pages} TIFF pages.")
    print(f"Selected channel {snake_channel_index} for the active contour.")
    print(f"Dynamic anchor frame interval: every {dynamic_anchor_frame_interval} frame(s)")
    print(f"Dynamic anchor frames (0-based CSV Frame values): {dynamic_anchor_frames.tolist()}")

    refined_snake_rc = active_contour(
        snake_image,
        initial_snake_rc,
        **snake_active_contour_kwargs,
    )
    refined_snake_xy = np.column_stack([refined_snake_rc[:, 1], refined_snake_rc[:, 0]])
    refined_polyline = pd.DataFrame(refined_snake_xy, columns=["X", "Y"])

    df_spots = pd.read_csv(spots_path, skiprows=[1, 2, 3])
    if "DATA_INPUT" in df_spots.columns:
        df_spots["DATA_INPUT"] = data_input
    else:
        df_spots.insert(0, "DATA_INPUT", data_input)

    static_curve_cache = build_curve_cache(refined_snake_xy)
    static_curve = static_curve_cache["curve"]
    spot_xy = df_spots[["X", "Y"]].to_numpy(dtype=float)

    static_results = measure_points_against_curve(spot_xy, static_curve_cache)
    if len(static_results) > 0:
        static_closest_boundary_xy = np.vstack(static_results[:, 1])
        df_spots["signed_distance_to_boundary"] = static_results[:, 0].astype(float)
        df_spots["closest_boundary_x"] = static_closest_boundary_xy[:, 0]
        df_spots["closest_boundary_y"] = static_closest_boundary_xy[:, 1]
        df_spots["closest_boundary_segment"] = static_results[:, 2].astype(int)
        df_spots["boundary_arc_length"] = static_results[:, 3].astype(float)
    else:
        df_spots["signed_distance_to_boundary"] = np.array([], dtype=float)
        df_spots["closest_boundary_x"] = np.array([], dtype=float)
        df_spots["closest_boundary_y"] = np.array([], dtype=float)
        df_spots["closest_boundary_segment"] = np.array([], dtype=int)
        df_spots["boundary_arc_length"] = np.array([], dtype=float)

    dynamic_contour_cache_metadata = build_dynamic_contour_cache_metadata(
        data_input,
        image_path,
        polyline_path,
        snake_channel_index,
        n_channels,
        n_timepoints_for_processing,
        dynamic_anchor_frame_interval,
        dynamic_anchor_frames,
        snake_active_contour_kwargs,
        initial_snake_xy,
    )
    dynamic_contour_cache_file = dynamic_contour_cache_path(
        config.get("dynamic_contour_cache_dir", DYNAMIC_CONTOUR_CACHE_DIR),
        dynamic_contour_cache_metadata,
    )
    cached_dynamic_anchor_curves = None

    if config.get("use_dynamic_contour_cache", True):
        cached_dynamic_anchor_curves = load_dynamic_anchor_curves_from_cache(dynamic_contour_cache_file, dynamic_anchor_frames)

    if cached_dynamic_anchor_curves is not None:
        dynamic_anchor_curves = cached_dynamic_anchor_curves
        dynamic_anchor_curve_caches = {
            anchor_frame: build_curve_cache(anchor_curve) for anchor_frame, anchor_curve in dynamic_anchor_curves.items()
        }
        print(f"Loaded dynamic anchor contours from cache: {dynamic_contour_cache_file}")
    else:
        if config.get("use_dynamic_contour_cache", True):
            print(f"No matching dynamic contour cache found; computing contours for: {image_path.name}")

        dynamic_anchor_curves = {0: refined_snake_xy.copy()}
        dynamic_anchor_curve_caches = {0: build_curve_cache(refined_snake_xy)}
        previous_anchor_snake_rc = refined_snake_rc.copy()

        for anchor_frame in dynamic_anchor_frames[1:]:
            anchor_source = load_channel_frame(image_path, snake_channel_index, anchor_frame, n_channels)
            anchor_image = preprocess_snake_source(anchor_source)
            anchor_snake_rc = active_contour(
                anchor_image,
                previous_anchor_snake_rc,
                **snake_active_contour_kwargs,
            )
            anchor_snake_xy = np.column_stack([anchor_snake_rc[:, 1], anchor_snake_rc[:, 0]])
            dynamic_anchor_curves[anchor_frame] = anchor_snake_xy
            dynamic_anchor_curve_caches[anchor_frame] = build_curve_cache(anchor_snake_xy)
            previous_anchor_snake_rc = anchor_snake_rc

        if config.get("use_dynamic_contour_cache", True):
            save_dynamic_anchor_curves_to_cache(
                dynamic_contour_cache_file,
                dynamic_contour_cache_metadata,
                dynamic_anchor_frames,
                dynamic_anchor_curves,
            )
            print(f"Saved dynamic anchor contours to cache: {dynamic_contour_cache_file}")

    df_spots_dynamic = df_spots.copy()
    spot_anchor_frames = np.array(
        [
            assign_dynamic_anchor_frame(frame_value, dynamic_anchor_frames)
            for frame_value in df_spots_dynamic["Frame"].to_numpy(dtype=int)
        ],
        dtype=int,
    )

    dynamic_signed_distance = np.full(len(df_spots_dynamic), np.nan, dtype=float)
    dynamic_closest_boundary_xy = np.full((len(df_spots_dynamic), 2), np.nan, dtype=float)
    dynamic_closest_boundary_segment = np.full(len(df_spots_dynamic), -1, dtype=int)
    dynamic_boundary_arc_length = np.full(len(df_spots_dynamic), np.nan, dtype=float)

    for anchor_frame in dynamic_anchor_frames:
        mask = spot_anchor_frames == anchor_frame
        anchor_points = df_spots_dynamic.loc[mask, ["X", "Y"]].to_numpy(dtype=float)
        anchor_results = measure_points_against_curve(anchor_points, dynamic_anchor_curve_caches[anchor_frame])
        if len(anchor_results) == 0:
            continue

        dynamic_signed_distance[mask] = anchor_results[:, 0].astype(float)
        dynamic_closest_boundary_xy[mask] = np.vstack(anchor_results[:, 1])
        dynamic_closest_boundary_segment[mask] = anchor_results[:, 2].astype(int)
        dynamic_boundary_arc_length[mask] = anchor_results[:, 3].astype(float)

    df_spots_dynamic["dynamic_anchor_frame"] = spot_anchor_frames
    df_spots_dynamic["dynamic_signed_distance_to_boundary"] = dynamic_signed_distance
    df_spots_dynamic["dynamic_closest_boundary_x"] = dynamic_closest_boundary_xy[:, 0]
    df_spots_dynamic["dynamic_closest_boundary_y"] = dynamic_closest_boundary_xy[:, 1]
    df_spots_dynamic["dynamic_closest_boundary_segment"] = dynamic_closest_boundary_segment
    df_spots_dynamic["dynamic_boundary_arc_length"] = dynamic_boundary_arc_length

    if df_spots_dynamic[
        [
            "dynamic_signed_distance_to_boundary",
            "dynamic_closest_boundary_x",
            "dynamic_closest_boundary_y",
            "dynamic_boundary_arc_length",
        ]
    ].isna().any().any():
        raise ValueError("Dynamic boundary measurement left missing values in the spot table.")

    validate_dynamic_boundary_assignments(
        df_spots_dynamic,
        dynamic_anchor_frames,
        dynamic_anchor_curve_caches,
    )

    dynamic_anchor_summary = pd.DataFrame(
        {
            "anchor_frame": dynamic_anchor_frames,
            "contour_point_count": [len(dynamic_anchor_curve_caches[anchor_frame]["curve"]) for anchor_frame in dynamic_anchor_frames],
            "spot_count": [int((spot_anchor_frames == anchor_frame).sum()) for anchor_frame in dynamic_anchor_frames],
        }
    )

    dynamic_anchor_assignment_check = pd.DataFrame(
        {"frame": build_dynamic_anchor_assignment_check_frames(dynamic_anchor_frames, n_timepoints_for_processing)}
    )
    dynamic_anchor_assignment_check["assigned_anchor_frame"] = dynamic_anchor_assignment_check["frame"].apply(
        lambda frame_value: assign_dynamic_anchor_frame(frame_value, dynamic_anchor_frames)
    )

    dynamic_track_motion = compute_dynamic_track_motion(df_spots_dynamic)

    result = {
        "data_input": data_input,
        "config": config,
        "image_path": image_path,
        "polyline_path": polyline_path,
        "spots_path": spots_path,
        "n_pages": n_pages,
        "n_channels": n_channels,
        "n_timepoints": n_timepoints,
        "n_timepoints_for_processing": n_timepoints_for_processing,
        "channel_images_t0": channel_images_t0,
        "channel_display_ranges": channel_display_ranges,
        "channel_stats": channel_stats,
        "snake_channel_index": snake_channel_index,
        "fluorescence_channel_index": fluorescence_channel_index,
        "composite_rgb": composite_rgb,
        "polyline_xy": polyline_xy,
        "initial_snake_xy": initial_snake_xy,
        "refined_snake_xy": refined_snake_xy,
        "refined_polyline": refined_polyline,
        "static_curve": static_curve,
        "df_spots": df_spots,
        "df_spots_dynamic": df_spots_dynamic,
        "dynamic_anchor_frames": dynamic_anchor_frames,
        "dynamic_anchor_curves": dynamic_anchor_curves,
        "dynamic_anchor_curve_caches": dynamic_anchor_curve_caches,
        "dynamic_anchor_summary": dynamic_anchor_summary,
        "dynamic_anchor_assignment_check": dynamic_anchor_assignment_check,
        "dynamic_contour_cache_file": dynamic_contour_cache_file,
        "dynamic_track_links": dynamic_track_motion["dynamic_track_links"],
        "dynamic_radial_speed_summary": dynamic_track_motion["dynamic_radial_speed_summary"],
        "dynamic_tangential_speed_summary": dynamic_track_motion["dynamic_tangential_speed_summary"],
        "dynamic_track_motion_summary": dynamic_track_motion["dynamic_track_motion_summary"],
    }

    if show_plots:
        plot_dynamic_distance_result(result)

    return result


In [ ]:
processed_results = {}
skipped_data_inputs = []

for DATA_INPUT in DATA_INPUTS:
    if DATA_INPUT not in DATA_INPUT_CONFIGS:
        raise NotImplementedError(f"Data input '{DATA_INPUT}' is not recognized.")

    config = DATA_INPUT_CONFIGS[DATA_INPUT]
    missing_paths = missing_required_paths(config)
    if missing_paths:
        print_missing_file_warning(DATA_INPUT, missing_paths)
        skipped_data_inputs.append(
            {
                "data_input": DATA_INPUT,
                "status": "skipped_missing_file",
                "missing_files": "; ".join(str(path) for _, path in missing_paths),
            }
        )
        continue

    try:
        processed_results[DATA_INPUT] = process_data_input(
            DATA_INPUT,
            config,
            show_plots=SHOW_PLOTS,
            testing_mode=TESTING_MODE,
        )
    except FileNotFoundError as exc:
        missing_path = exc.filename if exc.filename else str(exc)
        print(f"Warning: skipping {DATA_INPUT}; file not found: {missing_path}")
        skipped_data_inputs.append(
            {
                "data_input": DATA_INPUT,
                "status": "skipped_missing_file",
                "missing_files": str(missing_path),
            }
        )
        continue

processing_summary_rows = []
for data_input, result in processed_results.items():
    processing_summary_rows.append(
        {
            "data_input": data_input,
            "status": "processed",
            "missing_files": "",
            "spot_count": len(result["df_spots_dynamic"]),
            "n_channels": result["n_channels"],
            "n_timepoints": result["n_timepoints"],
            "dynamic_anchor_count": len(result["dynamic_anchor_frames"]),
            "dynamic_link_count": len(result["dynamic_track_links"]),
            "dynamic_track_count": result["dynamic_track_motion_summary"][["DATA_INPUT", "Track ID"]].drop_duplicates().shape[0],
        }
    )

for skipped in skipped_data_inputs:
    processing_summary_rows.append(
        {
            "data_input": skipped["data_input"],
            "status": skipped["status"],
            "missing_files": skipped["missing_files"],
            "spot_count": 0,
            "n_channels": np.nan,
            "n_timepoints": np.nan,
            "dynamic_anchor_count": 0,
            "dynamic_link_count": 0,
            "dynamic_track_count": 0,
        }
    )

processing_summary = pd.DataFrame(processing_summary_rows)

if processed_results:
    combined_static_spots = pd.concat(
        [result["df_spots"] for result in processed_results.values()],
        ignore_index=True,
    )
    combined_dynamic_spots = pd.concat(
        [result["df_spots_dynamic"] for result in processed_results.values()],
        ignore_index=True,
    )
    combined_dynamic_track_links = pd.concat(
        [result["dynamic_track_links"] for result in processed_results.values()],
        ignore_index=True,
    )
    combined_dynamic_radial_speed_summary = pd.concat(
        [result["dynamic_radial_speed_summary"] for result in processed_results.values()],
        ignore_index=True,
    )
    combined_dynamic_tangential_speed_summary = pd.concat(
        [result["dynamic_tangential_speed_summary"] for result in processed_results.values()],
        ignore_index=True,
    )
    combined_dynamic_track_motion_summary = pd.concat(
        [result["dynamic_track_motion_summary"] for result in processed_results.values()],
        ignore_index=True,
    )

    DATA_INPUT = next(reversed(processed_results))
    last_result = processed_results[DATA_INPUT]
    df_spots = last_result["df_spots"]
    df_spots_dynamic = last_result["df_spots_dynamic"]
    dynamic_track_links = last_result["dynamic_track_links"]
    dynamic_radial_speed_summary = last_result["dynamic_radial_speed_summary"]
    dynamic_tangential_speed_summary = last_result["dynamic_tangential_speed_summary"]
    dynamic_track_motion_summary = last_result["dynamic_track_motion_summary"]
else:
    combined_static_spots = pd.DataFrame()
    combined_dynamic_spots = pd.DataFrame()
    combined_dynamic_track_links = pd.DataFrame()
    combined_dynamic_radial_speed_summary = pd.DataFrame()
    combined_dynamic_tangential_speed_summary = pd.DataFrame()
    combined_dynamic_track_motion_summary = pd.DataFrame()
    DATA_INPUT = None
    last_result = None
    dynamic_track_links = pd.DataFrame()
    dynamic_radial_speed_summary = pd.DataFrame()
    dynamic_tangential_speed_summary = pd.DataFrame()
    dynamic_track_motion_summary = pd.DataFrame()

print(f"Finished. Processed {len(processed_results)} DATA_INPUT option(s); skipped {len(skipped_data_inputs)}.")
processing_summary


In [ ]:
cell_median_columns = [
    "DATA_INPUT",
    "Track ID",
    "median_dynamic_radial_speed_um_per_s",
    "median_dynamic_tangential_speed_um_per_s",
]

if combined_dynamic_track_motion_summary.empty:
    cell_median_dynamic_speed_plot_data = add_treatment_plot_columns(pd.DataFrame(columns=cell_median_columns))
else:
    cell_median_dynamic_speed_plot_data = add_treatment_plot_columns(
        combined_dynamic_track_motion_summary[cell_median_columns]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "median_dynamic_radial_speed_um_per_s",
                "median_dynamic_tangential_speed_um_per_s",
            ]
        )
        .copy()
    )


def build_data_input_timepoint_speed_plot_data(dynamic_track_links, speed_column, mean_column):
    output_columns = ["DATA_INPUT", "T", mean_column, "cell_count", "link_count"]
    if dynamic_track_links.empty:
        return add_treatment_plot_columns(pd.DataFrame(columns=output_columns))

    required_columns = ["DATA_INPUT", "Track ID", "Spot ID", "T", speed_column]
    missing_columns = [column for column in required_columns if column not in dynamic_track_links.columns]
    if missing_columns:
        raise ValueError(f"Cannot build timepoint plot data; missing columns: {missing_columns}")

    finite_links = (
        dynamic_track_links[required_columns]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["T", speed_column])
        .copy()
    )
    if finite_links.empty:
        return add_treatment_plot_columns(pd.DataFrame(columns=output_columns))

    grouped = finite_links.groupby(["DATA_INPUT", "T"])
    timepoint_speed = grouped[speed_column].mean().reset_index(name=mean_column)
    cell_counts = grouped["Track ID"].nunique().reset_index(name="cell_count")
    link_counts = grouped["Spot ID"].count().reset_index(name="link_count")

    return add_treatment_plot_columns(
        timepoint_speed.merge(cell_counts, on=["DATA_INPUT", "T"], how="left")
        .merge(link_counts, on=["DATA_INPUT", "T"], how="left")
        .sort_values(["DATA_INPUT", "T"])
        .reset_index(drop=True)
    )


data_input_timepoint_radial_speed_plot_data = build_data_input_timepoint_speed_plot_data(
    combined_dynamic_track_links,
    "dynamic_radial_speed_um_per_s",
    "mean_dynamic_radial_speed_um_per_s",
)
data_input_timepoint_tangential_speed_plot_data = build_data_input_timepoint_speed_plot_data(
    combined_dynamic_track_links,
    "dynamic_tangential_speed_um_per_s",
    "mean_dynamic_tangential_speed_um_per_s",
)

plot_cell_median_dynamic_speeds(cell_median_dynamic_speed_plot_data)
plot_data_input_timepoint_speed(
    data_input_timepoint_radial_speed_plot_data,
    "mean_dynamic_radial_speed_um_per_s",
    "Mean signed dynamic radial speed (um/s)",
    "Mean Dynamic Radial Speed by DATA_INPUT and Time",
    "mean dynamic radial speed time plot",
)
plot_data_input_timepoint_speed(
    data_input_timepoint_tangential_speed_plot_data,
    "mean_dynamic_tangential_speed_um_per_s",
    "Mean dynamic tangential speed (um/s)",
    "Mean Dynamic Tangential Speed by DATA_INPUT and Time",
    "mean dynamic tangential speed time plot",
)

cell_median_dynamic_speed_plot_data.head()
